# 常驻与同源：systemd 与 Nginx 反向代理

[原网页：模块 7.2](https://李勃老师.com/zero-to-fullstack/lessons/module-7-2/)

## 朴素的部署留下的三笔账

上一节已经把项目部署上线，但还留下三个问题：

- 后端服务脆弱：服务器重启或程序崩溃后，`nohup` 不会自动把它拉起来。
- 8000 端口直接暴露公网：后端没有经过 Nginx 统一处理。
- 前后端不同源：前端走 80，后端走 8000，仍然受跨源规则限制。

这一节逐一解决：用 systemd 管后端，用 Nginx 反向代理接住 API，再让前端使用同源的相对路径。

## 第一笔：systemd 取代 nohup

上一节从前台运行：

~~~bash
fastapi run
~~~

改成后台运行：

~~~bash
nohup .venv/bin/fastapi run > backend.log 2>&1 &
~~~

`nohup` 让程序不再依赖一直开着的 SSH 会话，但日常操作还很麻烦。想知道进程是否存在，先查：

~~~bash
ps aux | grep fastapi
~~~

输出示例：

~~~text
ubuntu   3701149  0.1 13.2 774360 506288 ?   Sl   Sep19  29:46 /home/ubuntu/zero-to-tech/backend/.venv/bin/python3 .venv/bin/fastapi run
~~~

以空格分列，第二列是 PID。可能同时查到 `grep` 自己，要找到真正运行 Python/FastAPI 的那一行。停掉旧后端时，使用它的进程号：

~~~bash
kill 那个进程号
~~~

再查询一次，确认旧进程已经停止。这时文字实验室暂时不能分析；接下来交给 systemd 启动同一个后端。

`nohup` 也不能解决开机启动和崩溃重启。我们需要一个“管家”，负责启动、守护、状态查询和日志管理。Ubuntu 上的这个工具就是 **systemd**，让程序持续被看护的做法叫进程守护。

名字末尾的 `d` 来自 daemon（守护进程），类似的程序还有 `httpd`、`crond`。

### 把程序委托给 systemd

我们此前管理 Nginx 时已经使用过 systemd：

~~~bash
systemctl status nginx
~~~

~~~bash
sudo systemctl reload nginx
~~~

`systemctl` 是 systemd 的命令行工具；`nginx` 是服务名，`status`、`reload` 是交给 systemd 的指令。

Nginx 自己也能通过这些命令启动、重载和停止：

~~~bash
sudo nginx              # 启动
sudo nginx -s reload    # 重载配置
sudo nginx -s stop      # 停
~~~

这里了解即可，当前 Nginx 已由 systemd 管理，不要再手工启动一份，否则可能撞端口。

两者的分工是：

- 程序自己的命令规定“怎样运行、怎样停止”。
- systemd 在这些命令之上负责开机启动、崩溃重启和日志管理。

### Nginx 是怎么委托的？

委托需要一份 **service 文件**，告诉 systemd 程序怎样运行、希望怎样管理。

查看 Nginx 的文件：

~~~bash
systemctl cat nginx
~~~

不同版本略有差异，整体形状类似：

~~~ini
# /lib/systemd/system/nginx.service
[Unit]
Description=A high performance web server and a reverse proxy server
...

[Service]
Type=forking
ExecStart=/usr/sbin/nginx -g 'daemon on; master_process on;'
ExecReload=/usr/sbin/nginx -g 'daemon on; master_process on;' -s reload
...

[Install]
WantedBy=multi-user.target
~~~

service 文件主要分三段：

| 段 | 回答的问题 |
| --- | --- |
| `[Unit]` | 这是什么服务、和其他服务有什么关系 |
| `[Service]` | 用什么命令和身份运行、进程退出后怎么办 |
| `[Install]` | 启用时与哪个启动目标关联 |

`ExecStart` 是拉起程序的命令；`WantedBy=multi-user.target` 用于后面启用开机自启。

内容一屏放不下时会进入分页器：方向键上下移动，空格向下翻页，`b` 向上翻页，`q` 退出。

### 如何写自己的 service 文件

在 systemd 的配置目录创建后端服务文件：

~~~bash
sudo vim /etc/systemd/system/zero-backend.service
~~~

写入网页示例：

~~~ini
[Unit]
Description=zero-to-tech backend

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/zero-to-tech/backend
ExecStart=/home/ubuntu/zero-to-tech/backend/.venv/bin/fastapi run
Restart=always

[Install]
WantedBy=multi-user.target
~~~

这里的 `ubuntu` 和 `/home/ubuntu/zero-to-tech` 要替换为服务器实际用户名和项目路径。

| 配置项 | 含义 |
| --- | --- |
| `Description` | 给人看的服务描述，查状态时会显示 |
| `User` | 以指定用户身份运行；不写默认使用 root，本项目使用普通用户即可 |
| `WorkingDirectory` | 工作目录，使用后端目录的绝对路径 |
| `ExecStart` | 启动命令，直接指定 `.venv` 中的 FastAPI 绝对路径 |
| `Restart=always` | 进程退出后由 systemd 重新启动 |
| `WantedBy=multi-user.target` | 配合 `enable`，设置开机时启动 |

这里不需要先 `activate`。激活环境只是帮助终端优先找到 `.venv` 中的命令，绝对路径已经明确指定了使用哪一份 FastAPI。

你在 7.1 中用 uv 创建的 `.venv` 同样可以由这里的绝对路径调用。

### 通过 service 文件启用委托

文件保存后依次执行：

~~~bash
sudo systemctl daemon-reload          # 新增了 service 文件，让 systemd 重新读一遍
sudo systemctl start zero-backend     # 启动
sudo systemctl status zero-backend    # 查状态
sudo systemctl enable zero-backend    # 开机自启（就是 WantedBy 那行生效）
~~~

四条命令的作用是：

1. `daemon-reload`：让 systemd 重新读取新增的 service 文件。
2. `start`：立即启动服务。
3. `status`：查询服务状态。
4. `enable`：启用开机自启。

状态示例：

~~~text
● zero-backend.service - zero-to-tech backend
     Loaded: loaded (/etc/systemd/system/zero-backend.service; enabled; ...)
     Active: active (running) since ...
   Main PID: 12345 (fastapi)
~~~

看到 `Active: active (running)`，说明服务已启动。打开文字实验室，分析功能应当恢复。

### 验证崩溃后会重新启动

从 `status` 输出中找到 `Main PID`，模拟进程退出：

~~~bash
sudo kill 进程号          # 换成你看到的 Main PID
~~~

随后再次查状态：

~~~bash
sudo systemctl status zero-backend
~~~

应该仍然看到 `active (running)`，但 `Main PID` 变成了另一个号码。这说明旧进程已经结束，systemd 又启动了一个新进程。

这就是 `Restart=always` 带来的自动重启。

### 用 systemctl 指挥后端

委托之后，日常管理都使用 `systemctl`：

| 想做什么 | 命令 |
| --- | --- |
| 查看状态 | `sudo systemctl status zero-backend` |
| 修改后端代码后重启 | `sudo systemctl restart zero-backend` |
| 停止 | `sudo systemctl stop zero-backend` |
| 启动 | `sudo systemctl start zero-backend` |
| 开机自启 | `sudo systemctl enable zero-backend` |
| 取消开机自启 | `sudo systemctl disable zero-backend` |
| 实时查看日志 | `journalctl -u zero-backend -f` |

**`stop` 和 `kill` 不同**：通过 `systemctl stop` 是明确要求 systemd 停止服务，即使配置了 `Restart=always`，也不会自动拉起。

可以分别停止、启动，再调用接口验证：

~~~bash
sudo systemctl stop zero-backend
curl localhost:8000/api/profile
# curl: (7) Failed to connect ... Connection refused    ← 真的停了

sudo systemctl start zero-backend
curl localhost:8000/api/profile
# JSON 回来了
~~~

### 后端程序的三种日志方式

**终端实时显示**：本地 `fastapi dev` 占住终端，请求、`print` 和 traceback 都直接显示。窗口关闭后，之前显示的内容不再方便查看。

**重定向到文件**：上一节通过这段命令接住日志：

~~~bash
nohup .venv/bin/fastapi run > backend.log 2>&1 &
#                           └──────┬──────┘
#                         把输出接进 backend.log
~~~

`cat backend.log` 可以查看文件，`tail -f backend.log` 可以跟随新输出。但文件名、保存位置和清理都要自己管理。

**交给 systemd**：服务的标准输出和错误输出交给 systemd 日志系统。它会管理容量和旧日志的清理，不再需要手动维护 `backend.log`。

### 使用 journalctl 看日志

`journalctl` 是查看 systemd 日志的工具：

~~~bash
journalctl -u zero-backend            # 这个服务的日志，从头看
journalctl -u zero-backend -n 50      # 只看最后 50 行
journalctl -u zero-backend -f         # 实时跟随，新日志一出现就打出来
~~~

- `-u`：unit，指定查看哪个服务。
- `-n 50`：查看最后 50 行。
- `-f`：follow，持续等待新日志并显示。

亲自运行：

~~~bash
journalctl -u zero-backend -f
~~~

终端先显示最近几行，然后停在那里等待。切换到浏览器点一次“分析”，再回来就会看到新日志。

`Ctrl + C` 结束跟随，回到命令提示符。没有 `-f` 时，日志较多会进入分页器，按 `q` 退出。

### 清掉旧日志文件

确认新服务已由 systemd 管理后，`backend.log` 不再更新，留下它容易误以为里面是当前日志。网页中将它删除：

~~~bash
rm ~/zero-to-tech/backend/backend.log
~~~

此后统一通过 `journalctl -u zero-backend` 查询后端日志。

## 第二笔：Nginx 与反向代理

目前 Nginx 在 80 端口提供前端，Uvicorn 在 8000 端口运行后端，两个进程都由 systemd 管理。

它们都接收 HTTP 请求，但分工不同：

- **Nginx** 擅长对外提供文件、记录访问、处理 HTTPS、压缩和缓存。
- **Uvicorn** 负责把请求交给 Python/FastAPI，把业务计算结果返回。

Nginx 不会直接运行项目的 Python 业务代码，所以仍然需要 Uvicorn。接下来是让所有外部请求先到 Nginx，由它按路径决定是返回静态文件，还是转交 Uvicorn。

### 运行日志和访问日志

网页中比较 Nginx 和 Uvicorn 的对外能力：

| 能力 | Nginx | Uvicorn |
| --- | --- | --- |
| 访问日志 | 有专门的日志，可记录来源、浏览器、请求和字节数 | 本节没有单独配置同类访问日志 |
| HTTPS | 便于集中配置证书和续期 | 可以配置证书，但需要自行管理 |
| 压缩、缓存 | 有现成能力 | 本项目没有这些配套配置 |
| 对外入口 | 专门处理公网请求 | 主要职责是运行 Python 应用 |

这里的访问日志和 `journalctl` 所看的日志不同：

- **运行日志**：程序做了什么、有没有错误、traceback 是什么，由 systemd 收集。
- **访问日志（access log）**：谁请求了哪些页面、从哪里来、用什么浏览器等；Nginx 单独写在 `/var/log/nginx/access.log`。

两个服务继续存在：对外的工作交给 Nginx，业务计算交给 Uvicorn。

### 按路径分工

前端页面的路径是：

~~~text
/
/text-lab
~~~

后端接口的路径是：

~~~text
/api/profile
/api/analyze
/api/history
~~~

因此 Nginx 只需监听 80 端口：

- `/`、`/text-lab` 等页面请求，到静态目录里找文件。
- `/api/` 开头的请求，转交本机 Uvicorn 的 8000 端口。

Uvicorn 继续运行在 8000 上，后面会把它限制为只接受本机连接。

~~~text
浏览器 → Nginx（80）
          ├─ 页面路径 → out/ 静态文件
          └─ /api/   → Uvicorn（本机 8000）→ FastAPI
~~~

### 配置反向代理

打开 Nginx 的站点配置：

~~~bash
sudo vim /etc/nginx/sites-enabled/default
~~~

在现有 `server` 块中增加 `location /api/`，其他规则保持原有设置。网页中的完整示例：

~~~nginx
server {
    listen 80 default_server;
    server_name _;

    root /home/ubuntu/zero-to-tech/out;
    index index.html;

    location / {
        try_files $uri $uri.html $uri/ =404;
    }

    location /api/ {                          # ← 新增：路径以 /api/ 开头的请求
        proxy_pass http://127.0.0.1:8000;     # ← 别找文件了，原样转给本机 8000

        proxy_set_header Host              $host;              # 访客敲的是哪个地址
        proxy_set_header X-Real-IP         $remote_addr;       # 访客的真实 IP
        proxy_set_header X-Forwarded-For   $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;            # 访客走的是 http 还是 https
    }
}
~~~

`location /api/` 告诉 Nginx：这些请求不再查找磁盘文件，转发到 `127.0.0.1:8000`，收到响应后返回给浏览器。

`location /` 仍然负责前端静态资源。`root` 应使用服务器实际的 `out/` 路径。

### proxy_pass 末尾不要加斜杠

本项目的正确写法是：

~~~nginx
proxy_pass http://127.0.0.1:8000;
~~~

不能改成：

~~~nginx
proxy_pass http://127.0.0.1:8000/;
~~~

在 `location /api/` 下，不带末尾斜杠会保留原来的 `/api/...` 路径；带斜杠则会替换匹配的前缀。本项目 FastAPI 的接口本来带有 `/api/`，去掉它就会导致 404。

只有 `/api/` 会进入后端，`/docs` 等调试入口会落到静态资源规则；没有对应静态文件时返回 404，因此不再通过这个公网入口暴露。

### 为什么要写 proxy_set_header？

经过 Nginx 转发，后端不能只根据这一跳的连接判断原来的访客信息，所以 Nginx 要把信息附在请求头里一起传递：

| 请求头 | Nginx 变量 | 传递的信息 |
| --- | --- | --- |
| `Host` | `$host` | 访客请求的主机名 |
| `X-Real-IP` | `$remote_addr` | Nginx 看到的客户端 IP |
| `X-Forwarded-For` | `$proxy_add_x_forwarded_for` | 请求经过的代理链路 |
| `X-Forwarded-Proto` | `$scheme` | 原请求使用 HTTP 还是 HTTPS |

这些只是普通请求头。`X-` 前缀是自定义请求头的一种历史约定。

**反向代理（reverse proxy）**就是站在服务器一侧，替服务接收并转发请求；站在客户端一侧替用户访问其他站点的代理，则叫正向代理。

### 检查配置并验证反向代理

先检查语法，通过后再让 Nginx 重读配置：

~~~bash
sudo nginx -t
sudo systemctl reload nginx
~~~

在服务器本机验证：

~~~bash
curl localhost/api/profile
~~~

注意 `localhost` 后面没有 `:8000`，HTTP 默认端口是 80。请求路径是：

~~~text
curl → Nginx（80）→ Uvicorn（8000）→ 原路返回 JSON
~~~

再从自己的电脑访问：

~~~text
http://服务器IP/api/profile
~~~

不带 8000 也能获取 JSON，说明 Nginx 已经完成反向代理。

### 让 8000 不再对外：后端只听本机

现在公网请求已经可以走 Nginx，后端就不需要再监听所有网卡。

修改 service 文件：

~~~bash
sudo vim /etc/systemd/system/zero-backend.service
~~~

给 `ExecStart` 增加 `--host 127.0.0.1`：

~~~ini
ExecStart=/home/ubuntu/zero-to-tech/backend/.venv/bin/fastapi run --host 127.0.0.1
~~~

更新 service 后，重新加载配置并重启：

~~~bash
sudo systemctl daemon-reload      # service 文件改了，让 systemd 重新读一遍
sudo systemctl restart zero-backend
~~~

此时：

- 服务器本机 `curl localhost/api/profile` 仍能返回 JSON，这是走 Nginx。
- 服务器本机 `curl localhost:8000/api/profile` 也能返回 JSON，这是直接访问后端。
- 自己电脑访问 `http://服务器IP:8000/api/profile` 应当连接失败。

这一步之后，前端还在请求旧的公网 8000 地址，文字实验室会暂时无法分析；下一步修改前端配置后恢复。

### 删除安全组中 8000 的放行规则

到云平台控制台，删除安全组或防火墙中允许公网访问 8000 的规则。

8000 并没有停止使用：Uvicorn 仍在本机 8000 上运行，Nginx 可以访问它。限制的只是公网直接访问。

两层控制分别是：

1. Uvicorn 只监听 `127.0.0.1`。
2. 云平台安全组也不再放行 8000。

网页把这种多道防线称为纵深防御。

此时前后端都可通过同样的协议（HTTP）、主机（服务器 IP）、端口（80）访问，已经具备同源条件。

## 第三笔：前端切生产，把地址留空

服务器根目录中的 `.env.production` 仍然是旧配置：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://服务器IP:8000
~~~

去掉端口，改成下面这样也能运行：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://服务器IP
~~~

网页推荐直接把值留空：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=
~~~

`API` 变为空字符串后，代码中的 `${API}/api/profile` 就变成了 `/api/profile`。浏览器会根据当前页面补上协议、主机和端口。

因为已经同源，这个相对路径会访问正确的后端。以后从 IP 换成域名、从 HTTP 换成 HTTPS，也不必再改这个值。

修改生产配置后，重新构建前端：

~~~bash
cd ~/zero-to-tech
npm run build
~~~

静态前端需要重新构建，新的配置才会进入构建产物。

## 验证成果：同源

再次打开 `http://服务器IP/`，确认分析、拼音和历史记录仍能正常使用。

按 `F12` 打开 Network：

- API 请求和页面地址使用相同的协议、主机和端口。
- 请求不再发往 `:8000`。
- 点“分析”时，直接看到 POST 请求，没有跨源预检的 OPTIONS 请求。

**保留 `CORSMiddleware` 和 `credentials: "include"`**。线上同源后不再需要它们处理跨源请求，但本地开发仍然是 3000 → 8000 的跨源访问。

后端 `backend/.env` 中的 `ALLOWED_ORIGINS` 也保留。线上目前没有跨源请求需要它，但现有读取代码仍需要这个配置存在。

### 验证 8000 关闭和自动恢复

从自己的电脑访问：

~~~text
http://服务器IP:8000/api/profile
~~~

应该连接失败，而 `http://服务器IP/api/profile` 仍然正常。

再验证 systemd：

- 关闭 SSH 窗口，网站继续运行。
- 结束后端进程，systemd 自动启动新进程。
- 服务器重启后，已经 `enable` 的服务自动启动。

同时用不同浏览器或无痕窗口分析不同内容，确认每个会话仍只看到自己的历史记录。

## 回头把 README 改了

部署方式变化后，README 也要跟着更新。网页要求修改两处：

1. 重写“部署到服务器”，改为 systemd 和 Nginx 反向代理，并补充日常发布、运维命令。
2. 修改前端“配置说明”，把线上的 `NEXT_PUBLIC_API_BASE_URL` 改为留空。

下一单元保留网页完整 README 模板。模板中的后端环境准备使用网页原有的 `venv` 和 `requirements.txt` 写法；当前项目采用 uv 时，环境准备沿用 7.1 的 `uv sync`，systemd 仍直接运行同一个 `.venv/bin/fastapi`。

### 网页中的 README 部署模板

~~~~markdown
## 部署到服务器

前提：服务器上已装好 Python 3.10+、Node.js 18+ 和 Nginx。

**1. 拉取代码**

```bash
cd ~/zero-to-tech
git pull
```

**2. 前端：装依赖、写配置、构建**

```bash
npm install
cp .env.example .env.production   # 按下面「配置说明」填好（线上留空）
npm run build                     # 产物进 out/，由 Nginx 提供服务
```

**3. 后端：建环境、装依赖、写配置**

```bash
cd backend
python3 -m venv --prompt=zero-to-tech .venv   # 首次部署才需要
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env              # 按下面「配置说明」填好
```

**4. 后端：委托给 systemd**（首次部署才需要）

新建 `/etc/systemd/system/zero-backend.service`：

```ini
[Unit]
Description=zero-to-tech backend

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/zero-to-tech/backend
ExecStart=/home/ubuntu/zero-to-tech/backend/.venv/bin/fastapi run --host 127.0.0.1
Restart=always

[Install]
WantedBy=multi-user.target
```

```bash
sudo systemctl daemon-reload
sudo systemctl start zero-backend     # 启动
sudo systemctl enable zero-backend    # 开机自启
sudo systemctl status zero-backend    # 看到 active (running) 就对了
```

**5. Nginx：静态文件 ＋ `/api/` 反向代理**（首次部署才需要）

编辑 `/etc/nginx/sites-enabled/default`：

```nginx
server {
    listen 80 default_server;
    server_name _;

    root /home/ubuntu/zero-to-tech/out;
    index index.html;

    location / {
        try_files $uri $uri.html $uri/ =404;
    }

    location /api/ {
        proxy_pass http://127.0.0.1:8000;     # 末尾不要加斜杠

        proxy_set_header Host              $host;
        proxy_set_header X-Real-IP         $remote_addr;
        proxy_set_header X-Forwarded-For   $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}
```

```bash
sudo nginx -t && sudo systemctl reload nginx
```

**6. 安全组 / 防火墙**

只放行 **22（SSH）** 和 **80（HTTP）**。**8000 不要放行**——后端只监听 `127.0.0.1`，由 Nginx 转发进去。

**7. 验证**

```bash
curl localhost/api/profile        # 服务器本机，注意不带端口
```

浏览器访问 `http://服务器IP`，做一次分析、看一眼历史记录。换个无痕窗口再试，两边的历史应该互相看不到。

## 日常发布

```bash
cd ~/zero-to-tech
git pull

npm run build                          # 前端改了才需要
sudo systemctl restart zero-backend    # 后端改了才需要
```

## 常用运维命令

```bash
sudo systemctl status zero-backend     # 看状态
sudo systemctl restart zero-backend    # 重启
sudo systemctl stop zero-backend       # 停掉
journalctl -u zero-backend -f          # 实时看日志（Ctrl+C 退出）
journalctl -u zero-backend -n 50       # 看最后 50 行

sudo nginx -t                          # 改完 Nginx 配置先验语法
sudo systemctl reload nginx            # 再让它重读
```
~~~~

### 更新配置说明

README 中的前端配置表改为：

| 键 | 说明 | 本地 | 线上 |
| --- | --- | --- | --- |
| `NEXT_PUBLIC_API_BASE_URL` | 后端接口地址 | `http://localhost:8000` | 留空，使用同源相对路径 |

后端 `ALLOWED_ORIGINS` 不用更改，仍填写：

~~~dotenv
ALLOWED_ORIGINS=http://服务器IP
~~~

现有代码使用 `os.getenv("ALLOWED_ORIGINS").split(",")`；缺少配置会导致启动报错，所以不能因为线上同源就把这一行清空或删除。

更新 README 后提交：

~~~bash
git add README.md
git commit -m "更新部署说明：systemd ＋ 反向代理 ＋ 同源"
git push
~~~

代码和部署方式改变后，文档也要同步修改。

## 结尾

这一节完成了后端的常驻管理、Nginx 反向代理和前后端同源：

- systemd 管理启动、重启、开机自启和运行日志。
- Nginx 作为公网入口，静态文件和 API 按路径分工。
- Uvicorn 只监听本机，8000 不再向公网开放。
- 前端使用相对路径，以后换域名和 HTTPS 时不必重新填写后端地址。

现在还有两个问题：IP 地址不好记，HTTP 仍然是明文传输。下一节将配置域名和 HTTPS。